# 04 Combine, Global Clean, Balance, and Split Somali News Dataset

This notebook combines the final model-ready files for:

- `caafimaad`
- `ciyaaro`
- `siyaasad`

It performs final global checks, removes unsafe cross-category duplicates, rebuilds `input_text` and `target_text`, balances the categories, and creates final `train.csv`, `validation.csv`, and `test.csv` files for mT5 fine-tuning.

Expected input files in the same folder as this notebook:

```text
caafimaad_model_ready.csv
ciyaaro_model_ready.xlsx
siyaasad_model_ready.xlsx
```

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

## 1. Settings

In [2]:
RANDOM_STATE = 42

CAAFIMAAD_FILE = "ready_dataset/caafimaad_model_ready.csv"
CIYAARO_FILE = "ready_dataset/ciyaaro_model_ready.xlsx"
SIYAASAD_FILE = "ready_dataset/siyaasad_model_ready.xlsx"

OUTPUT_DIR = Path("final_combined_dataset")
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET_PER_CATEGORY = None

MIN_BODY_WORDS = 50
MIN_HEADLINE_WORDS = 3

TRAIN_RATIO = 0.80
VALIDATION_RATIO = 0.10
TEST_RATIO = 0.10

assert abs((TRAIN_RATIO + VALIDATION_RATIO + TEST_RATIO) - 1.0) < 1e-9

## 2. Helper functions

In [3]:
def read_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported file type: {path.suffix}")


def print_basic_info(name, df):
    print("=" * 90)
    print(name)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    if "category" in df.columns:
        print("\nCategory counts:")
        print(df["category"].value_counts(dropna=False))
    if "source" in df.columns:
        print("\nSource counts:")
        print(df["source"].value_counts(dropna=False))
    print("\nMissing values:")
    print(df.isna().sum())


def normalize_basic_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace("\\n", "\n")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\u00a0", " ")
    text = text.replace("\ufeff", "")
    text = text.replace("\u200b", "")
    text = text.replace("\u200c", "")
    text = text.replace("\u200d", "")
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def normalize_key(text):
    text = normalize_basic_text(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

## 3. Load the three category files

In [4]:
caafimaad = read_file(CAAFIMAAD_FILE)
ciyaaro = read_file(CIYAARO_FILE)
siyaasad = read_file(SIYAASAD_FILE)

print_basic_info("CAAFIMAAD", caafimaad)
print_basic_info("CIYAARO", ciyaaro)
print_basic_info("SIYAASAD", siyaasad)

CAAFIMAAD
Shape: (1134, 9)
Columns: ['url', 'source', 'category', 'headline_clean', 'body_clean', 'body_word_count_clean', 'headline_word_count', 'input_text', 'target_text']

Category counts:
category
caafimaad    1134
Name: count, dtype: int64

Source counts:
source
Goobjoog      642
BBC Somali    492
Name: count, dtype: int64

Missing values:
url                      0
source                   0
category                 0
headline_clean           0
body_clean               0
body_word_count_clean    0
headline_word_count      0
input_text               0
target_text              0
dtype: int64
CIYAARO
Shape: (1274, 23)
Columns: ['url', 'headline', 'body', 'category', 'source', 'status', 'body_word_count', 'scraped_at', 'error', 'headline_clean', 'body_clean', 'headline_word_count_raw', 'headline_word_count_clean', 'body_word_count_raw', 'body_word_count_clean', 'empty_headline_clean', 'empty_body_clean', 'short_body_clean', 'duplicate_url', 'duplicate_headline_clean', 'duplicate_bod

## 4. Standardize required columns

In [5]:
REQUIRED_COLUMNS = [
    "url",
    "source",
    "category",
    "headline_clean",
    "body_clean",
    "input_text",
    "target_text"
]

def standardize_dataset(df, expected_category):
    df = df.copy()
    for col in REQUIRED_COLUMNS:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")
    df = df[REQUIRED_COLUMNS].copy()
    for col in REQUIRED_COLUMNS:
        df[col] = df[col].fillna("").astype(str).str.strip()
    df["category"] = expected_category
    df["category_file"] = expected_category
    return df

caafimaad_std = standardize_dataset(caafimaad, "caafimaad")
ciyaaro_std = standardize_dataset(ciyaaro, "ciyaaro")
siyaasad_std = standardize_dataset(siyaasad, "siyaasad")

combined_raw = pd.concat(
    [caafimaad_std, ciyaaro_std, siyaasad_std],
    ignore_index=True
)

print("Combined raw shape:", combined_raw.shape)
print("\nCombined category counts:")
print(combined_raw["category"].value_counts())
print("\nCombined source counts:")
print(combined_raw["source"].value_counts())

Combined raw shape: (4093, 8)

Combined category counts:
category
siyaasad     1685
ciyaaro      1274
caafimaad    1134
Name: count, dtype: int64

Combined source counts:
source
Goobjoog      2518
BBC Somali     930
Laacibnet      645
Name: count, dtype: int64


## 5. Apply conservative global cleaning

In [6]:
GLOBAL_REMOVE_LINE_PATTERNS = [
    r"^Goobjoog News$",
    r"^Googjoog News$",
    r"^Dhageyso$",
    r"^Halkaan hoose ka dhageyso:?\s*$",
    r"^Halkaan ka Akhriso:?\s*$",
    r"^Halkan Ka Daawo.*$",
    r"^Sidoo Kale Aqriso.*$",
    r"^Save my name, email, and website in this browser for the next time I comment\.?$",
    r"^laacibnet$",
    r"^comments?$",
    r"^Leave a Reply$",
    r"^Cancel reply$",
    r"^Email-.*$",
    r"^W/D\s*[-–].*$",
    r"^W/Q\s*[-–].*$",
    r"^©\s*\d{4}.*$",
]

def remove_global_noise_lines(text):
    text = normalize_basic_text(text)
    cleaned_lines = []
    for raw_line in text.splitlines():
        line = normalize_basic_text(raw_line)
        if not line:
            continue
        remove = any(
            re.search(pattern, line, flags=re.IGNORECASE)
            for pattern in GLOBAL_REMOVE_LINE_PATTERNS
        )
        if not remove:
            cleaned_lines.append(line)
    cleaned = "\n".join(cleaned_lines)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
    cleaned = re.sub(r"[ \t]+", " ", cleaned)
    return cleaned.strip()


df = combined_raw.copy()

df["url"] = df["url"].apply(normalize_basic_text)
df["source"] = df["source"].apply(normalize_basic_text)
df["headline_clean"] = df["headline_clean"].apply(normalize_basic_text)
df["body_clean"] = df["body_clean"].apply(remove_global_noise_lines)

df["category"] = df["category"].str.lower().str.strip()
df["category"] = df["category"].replace({
    "cayaaraha": "ciyaaro",
    "ciyaaraha": "ciyaaro",
    "sports": "ciyaaro",
    "sport": "ciyaaro",
    "caafimaadka": "caafimaad",
    "saynis_iyo_caafimaad": "caafimaad",
    "siyaasadd": "siyaasad",
    "siyaasadda": "siyaasad",
})

print("After global normalization:")
print(df["category"].value_counts())

After global normalization:
category
siyaasad     1685
ciyaaro      1274
caafimaad    1134
Name: count, dtype: int64


## 6. Remove bad quality rows

In [7]:
df["body_word_count_clean"] = df["body_clean"].str.split().str.len()
df["headline_word_count_clean"] = df["headline_clean"].str.split().str.len()

df["empty_url"] = df["url"].str.strip().eq("")
df["empty_body_clean"] = df["body_clean"].str.strip().eq("")
df["empty_headline_clean"] = df["headline_clean"].str.strip().eq("")

BAD_HEADLINE_PATTERNS = [
    r"^No Headline Found$",
    r"^No Headline$",
    r"^Untitled$",
    r"^None$",
    r"^nan$",
]

def is_bad_headline(headline):
    headline = normalize_basic_text(headline)
    return any(re.search(pattern, headline, flags=re.IGNORECASE) for pattern in BAD_HEADLINE_PATTERNS)

df["bad_headline"] = df["headline_clean"].apply(is_bad_headline)
df["short_body"] = df["body_word_count_clean"] < MIN_BODY_WORDS
df["short_headline"] = df["headline_word_count_clean"] < MIN_HEADLINE_WORDS

bad_rows = df[
    df["empty_url"] |
    df["empty_body_clean"] |
    df["empty_headline_clean"] |
    df["bad_headline"] |
    df["short_body"] |
    df["short_headline"]
].copy()

print("Bad rows to remove:", len(bad_rows))
print("\nBad row reasons:")
print({
    "empty_url": int(df["empty_url"].sum()),
    "empty_body_clean": int(df["empty_body_clean"].sum()),
    "empty_headline_clean": int(df["empty_headline_clean"].sum()),
    "bad_headline": int(df["bad_headline"].sum()),
    "short_body": int(df["short_body"].sum()),
    "short_headline": int(df["short_headline"].sum()),
})

bad_rows.to_excel(OUTPUT_DIR / "01_removed_bad_quality_rows.xlsx", index=False)

df_quality = df[
    ~df["empty_url"] &
    ~df["empty_body_clean"] &
    ~df["empty_headline_clean"] &
    ~df["bad_headline"] &
    ~df["short_body"] &
    ~df["short_headline"]
].copy()

df_quality = df_quality.reset_index(drop=True)

print("\nAfter quality filtering:", df_quality.shape)
print(df_quality["category"].value_counts())

Bad rows to remove: 3

Bad row reasons:
{'empty_url': 0, 'empty_body_clean': 0, 'empty_headline_clean': 0, 'bad_headline': 1, 'short_body': 1, 'short_headline': 1}

After quality filtering: (4090, 16)
category
siyaasad     1683
ciyaaro      1274
caafimaad    1133
Name: count, dtype: int64


## 7. Cross-category duplicate analysis

In [8]:
df_quality["url_key"] = df_quality["url"].str.lower().str.strip()
df_quality["body_key"] = df_quality["body_clean"].apply(normalize_key)
df_quality["headline_key"] = df_quality["headline_clean"].apply(normalize_key)

duplicate_urls_all = df_quality[
    df_quality.duplicated("url_key", keep=False)
].sort_values(["url_key", "category", "source"])

duplicate_bodies_all = df_quality[
    df_quality.duplicated("body_key", keep=False)
].sort_values(["body_key", "category", "source"])

url_category_counts = (
    df_quality.groupby("url_key")["category"]
    .nunique()
    .reset_index(name="category_count")
)

conflict_url_keys = set(
    url_category_counts[url_category_counts["category_count"] > 1]["url_key"]
)

conflict_urls_removed = df_quality[
    df_quality["url_key"].isin(conflict_url_keys)
].sort_values(["url_key", "category", "source"])

body_category_counts = (
    df_quality.groupby("body_key")["category"]
    .nunique()
    .reset_index(name="category_count")
)

conflict_body_keys = set(
    body_category_counts[body_category_counts["category_count"] > 1]["body_key"]
)

conflict_bodies_removed = df_quality[
    df_quality["body_key"].isin(conflict_body_keys)
].sort_values(["body_key", "category", "source"])

print("Duplicate URL rows:", len(duplicate_urls_all))
print("Duplicate body rows:", len(duplicate_bodies_all))
print("Cross-category URL conflict rows:", len(conflict_urls_removed))
print("Cross-category body conflict rows:", len(conflict_bodies_removed))

duplicate_review_path = OUTPUT_DIR / "02_cross_category_duplicate_review.xlsx"

with pd.ExcelWriter(duplicate_review_path, engine="openpyxl") as writer:
    duplicate_urls_all.to_excel(writer, sheet_name="duplicate_urls_all", index=False)
    duplicate_bodies_all.to_excel(writer, sheet_name="duplicate_bodies_all", index=False)
    conflict_urls_removed.to_excel(writer, sheet_name="conflict_urls_removed", index=False)
    conflict_bodies_removed.to_excel(writer, sheet_name="conflict_bodies_removed", index=False)

print("Saved duplicate review:", duplicate_review_path)

Duplicate URL rows: 20
Duplicate body rows: 18
Cross-category URL conflict rows: 20
Cross-category body conflict rows: 18
Saved duplicate review: final_combined_dataset\02_cross_category_duplicate_review.xlsx


## 8. Remove unsafe cross-category conflicts

In [9]:
conflict_url_mask = df_quality["url_key"].isin(conflict_url_keys)
conflict_body_mask = df_quality["body_key"].isin(conflict_body_keys)

unsafe_conflicts = df_quality[conflict_url_mask | conflict_body_mask].copy()
unsafe_conflicts.to_excel(OUTPUT_DIR / "03_removed_unsafe_cross_category_conflicts.xlsx", index=False)

df_no_conflict = df_quality[~conflict_url_mask & ~conflict_body_mask].copy()
df_no_conflict = df_no_conflict.reset_index(drop=True)

print("Rows removed as unsafe conflicts:", len(unsafe_conflicts))
print("After removing unsafe conflicts:", df_no_conflict.shape)
print(df_no_conflict["category"].value_counts())

Rows removed as unsafe conflicts: 20
After removing unsafe conflicts: (4070, 19)
category
siyaasad     1673
ciyaaro      1269
caafimaad    1128
Name: count, dtype: int64


## 9. Remove remaining exact duplicate URL/body records

In [10]:
before = len(df_no_conflict)

df_dedup = df_no_conflict.drop_duplicates(subset=["url_key"], keep="first").copy()
after_url = len(df_dedup)

df_dedup = df_dedup.drop_duplicates(subset=["body_key"], keep="first").copy()
after_body = len(df_dedup)

df_dedup = df_dedup.reset_index(drop=True)

print("Before exact duplicate removal:", before)
print("After URL duplicate removal:", after_url)
print("After body duplicate removal:", after_body)
print("\nCategory counts after global cleaning:")
print(df_dedup["category"].value_counts())

Before exact duplicate removal: 4070
After URL duplicate removal: 4070
After body duplicate removal: 4070

Category counts after global cleaning:
category
siyaasad     1673
ciyaaro      1269
caafimaad    1128
Name: count, dtype: int64


## 10. Rebuild model fields

In [11]:
df_final_unbalanced = df_dedup.copy()

df_final_unbalanced["body_word_count_clean"] = df_final_unbalanced["body_clean"].str.split().str.len()
df_final_unbalanced["headline_word_count_clean"] = df_final_unbalanced["headline_clean"].str.split().str.len()

df_final_unbalanced["input_text"] = (
    "analyze somali news article: " + df_final_unbalanced["body_clean"]
)

df_final_unbalanced["target_text"] = (
    "category: " + df_final_unbalanced["category"] +
    " | headline: " + df_final_unbalanced["headline_clean"]
)

df_final_unbalanced = df_final_unbalanced.reset_index(drop=True)
df_final_unbalanced["id"] = [
    f"somali_news_{i+1:05d}" for i in range(len(df_final_unbalanced))
]

FINAL_COLUMNS = [
    "id",
    "url",
    "source",
    "category",
    "headline_clean",
    "body_clean",
    "body_word_count_clean",
    "headline_word_count_clean",
    "input_text",
    "target_text",
]

df_final_unbalanced = df_final_unbalanced[FINAL_COLUMNS].copy()

print("Final unbalanced shape:", df_final_unbalanced.shape)
print(df_final_unbalanced["category"].value_counts())

df_final_unbalanced.to_csv(
    OUTPUT_DIR / "04_somali_news_global_cleaned_unbalanced.csv",
    index=False,
    encoding="utf-8-sig"
)

df_final_unbalanced.to_excel(
    OUTPUT_DIR / "04_somali_news_global_cleaned_unbalanced.xlsx",
    index=False
)

print("Saved unbalanced global-cleaned files.")

Final unbalanced shape: (4070, 10)
category
siyaasad     1673
ciyaaro      1269
caafimaad    1128
Name: count, dtype: int64
Saved unbalanced global-cleaned files.


## 11. Balance categories by the smallest available category

In [17]:
# ============================================================
# 11. BALANCE CATEGORIES - SAFE VERSION
# ============================================================

category_counts = df_final_unbalanced["category"].value_counts()

print("Category counts before balancing:")
print(category_counts)

min_category_count = int(category_counts.min())

if TARGET_PER_CATEGORY is None:
    target_n = min_category_count
else:
    target_n = int(TARGET_PER_CATEGORY)
    if target_n > min_category_count:
        raise ValueError(
            f"TARGET_PER_CATEGORY={target_n} is larger than smallest category count={min_category_count}"
        )

print(f"\nBalancing each category to: {target_n} rows")

balanced_parts = []

for category_name, group in df_final_unbalanced.groupby("category", sort=False):
    sampled_group = group.sample(
        n=target_n,
        random_state=RANDOM_STATE
    ).copy()

    # Important: force category column to remain available
    sampled_group["category"] = category_name

    balanced_parts.append(sampled_group)

df_balanced = pd.concat(
    balanced_parts,
    ignore_index=True
)

# Shuffle final balanced dataset
df_balanced = df_balanced.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

# Recreate IDs after balancing
df_balanced["id"] = [
    f"somali_news_balanced_{i+1:05d}" for i in range(len(df_balanced))
]

# Check columns before selecting final columns
print("\nColumns after balancing:")
print(df_balanced.columns.tolist())

missing_columns = [col for col in FINAL_COLUMNS if col not in df_balanced.columns]

if missing_columns:
    raise ValueError(f"Missing columns after balancing: {missing_columns}")

df_balanced = df_balanced[FINAL_COLUMNS].copy()

print("\nBalanced shape:", df_balanced.shape)
print(df_balanced["category"].value_counts())

df_balanced.to_csv(
    OUTPUT_DIR / "05_somali_news_balanced_model_ready.csv",
    index=False,
    encoding="utf-8-sig"
)

df_balanced.to_excel(
    OUTPUT_DIR / "05_somali_news_balanced_model_ready.xlsx",
    index=False
)

print("Saved balanced model-ready files.")

Category counts before balancing:
category
siyaasad     1673
ciyaaro      1269
caafimaad    1128
Name: count, dtype: int64

Balancing each category to: 1128 rows

Columns after balancing:
['id', 'url', 'source', 'category', 'headline_clean', 'body_clean', 'body_word_count_clean', 'headline_word_count_clean', 'input_text', 'target_text']

Balanced shape: (3384, 10)
category
caafimaad    1128
ciyaaro      1128
siyaasad     1128
Name: count, dtype: int64
Saved balanced model-ready files.


## 12. Create train / validation / test splits

In [18]:
def split_each_category(group):
    group = group.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    n = len(group)
    n_test = int(round(n * TEST_RATIO))
    n_val = int(round(n * VALIDATION_RATIO))
    n_train = n - n_val - n_test

    train = group.iloc[:n_train].copy()
    validation = group.iloc[n_train:n_train+n_val].copy()
    test = group.iloc[n_train+n_val:].copy()

    train["split"] = "train"
    validation["split"] = "validation"
    test["split"] = "test"

    return train, validation, test


train_parts = []
validation_parts = []
test_parts = []

for category, group in df_balanced.groupby("category"):
    train_cat, validation_cat, test_cat = split_each_category(group)
    train_parts.append(train_cat)
    validation_parts.append(validation_cat)
    test_parts.append(test_cat)

train_df = pd.concat(train_parts, ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
validation_df = pd.concat(validation_parts, ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
test_df = pd.concat(test_parts, ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

train_df["id"] = [f"train_{i+1:05d}" for i in range(len(train_df))]
validation_df["id"] = [f"validation_{i+1:05d}" for i in range(len(validation_df))]
test_df["id"] = [f"test_{i+1:05d}" for i in range(len(test_df))]

print("Train shape:", train_df.shape)
print(train_df["category"].value_counts())

print("\nValidation shape:", validation_df.shape)
print(validation_df["category"].value_counts())

print("\nTest shape:", test_df.shape)
print(test_df["category"].value_counts())

Train shape: (2706, 11)
category
ciyaaro      902
caafimaad    902
siyaasad     902
Name: count, dtype: int64

Validation shape: (339, 11)
category
siyaasad     113
ciyaaro      113
caafimaad    113
Name: count, dtype: int64

Test shape: (339, 11)
category
siyaasad     113
ciyaaro      113
caafimaad    113
Name: count, dtype: int64


## 13. Save train / validation / test files

In [19]:
SPLIT_COLUMNS = FINAL_COLUMNS + ["split"]

train_df = train_df[SPLIT_COLUMNS].copy()
validation_df = validation_df[SPLIT_COLUMNS].copy()
test_df = test_df[SPLIT_COLUMNS].copy()

train_df.to_csv(
    OUTPUT_DIR / "train.csv",
    index=False,
    encoding="utf-8-sig"
)

validation_df.to_csv(
    OUTPUT_DIR / "validation.csv",
    index=False,
    encoding="utf-8-sig"
)

test_df.to_csv(
    OUTPUT_DIR / "test.csv",
    index=False,
    encoding="utf-8-sig"
)

train_df.to_excel(OUTPUT_DIR / "train.xlsx", index=False)
validation_df.to_excel(OUTPUT_DIR / "validation.xlsx", index=False)
test_df.to_excel(OUTPUT_DIR / "test.xlsx", index=False)

all_splits = pd.concat(
    [train_df, validation_df, test_df],
    ignore_index=True
)

all_splits.to_csv(
    OUTPUT_DIR / "06_somali_news_balanced_all_splits.csv",
    index=False,
    encoding="utf-8-sig"
)

all_splits.to_excel(
    OUTPUT_DIR / "06_somali_news_balanced_all_splits.xlsx",
    index=False
)

print("Saved train/validation/test files.")
print("\nAll splits shape:", all_splits.shape)

print("\nSplit distribution:")
print(pd.crosstab(all_splits["split"], all_splits["category"]))

print("\nFinal output folder:")
print(OUTPUT_DIR.resolve())

Saved train/validation/test files.

All splits shape: (3384, 11)

Split distribution:
category    caafimaad  ciyaaro  siyaasad
split                                   
test              113      113       113
train             902      902       902
validation        113      113       113

Final output folder:
E:\ml\final_combined_dataset


## 14. Final sanity checks

In [20]:
def final_checks(df, name):
    print("=" * 90)
    print(name)
    print("Shape:", df.shape)

    print("\nCategory counts:")
    print(df["category"].value_counts())

    print("\nMissing values:")
    print(df[["url", "headline_clean", "body_clean", "input_text", "target_text"]].isna().sum())

    print("\nEmpty checks:")
    print({
        "empty_url": int(df["url"].fillna("").astype(str).str.strip().eq("").sum()),
        "empty_headline_clean": int(df["headline_clean"].fillna("").astype(str).str.strip().eq("").sum()),
        "empty_body_clean": int(df["body_clean"].fillna("").astype(str).str.strip().eq("").sum()),
        "empty_input_text": int(df["input_text"].fillna("").astype(str).str.strip().eq("").sum()),
        "empty_target_text": int(df["target_text"].fillna("").astype(str).str.strip().eq("").sum()),
    })

    print("\nDuplicate checks:")
    print({
        "duplicate_url": int(df["url"].duplicated().sum()),
        "duplicate_body_clean": int(df["body_clean"].duplicated().sum()),
        "duplicate_input_text": int(df["input_text"].duplicated().sum()),
    })

    print("\nBody word count:")
    print(df["body_word_count_clean"].describe(percentiles=[.05, .25, .5, .75, .95]))

    print("\nHeadline word count:")
    print(df["headline_word_count_clean"].describe(percentiles=[.05, .25, .5, .75, .95]))


final_checks(df_final_unbalanced, "GLOBAL CLEANED UNBALANCED")
final_checks(df_balanced, "BALANCED FULL DATASET")
final_checks(train_df, "TRAIN")
final_checks(validation_df, "VALIDATION")
final_checks(test_df, "TEST")

GLOBAL CLEANED UNBALANCED
Shape: (4070, 10)

Category counts:
category
siyaasad     1673
ciyaaro      1269
caafimaad    1128
Name: count, dtype: int64

Missing values:
url               0
headline_clean    0
body_clean        0
input_text        0
target_text       0
dtype: int64

Empty checks:
{'empty_url': 0, 'empty_headline_clean': 0, 'empty_body_clean': 0, 'empty_input_text': 0, 'empty_target_text': 0}

Duplicate checks:
{'duplicate_url': 0, 'duplicate_body_clean': 0, 'duplicate_input_text': 0}

Body word count:
count    4070.000000
mean      337.297052
std       295.995087
min        50.000000
5%        106.450000
25%       154.000000
50%       213.000000
75%       450.250000
95%       908.000000
max      4348.000000
Name: body_word_count_clean, dtype: float64

Headline word count:
count    4070.000000
mean       12.118673
std         3.372204
min         3.000000
5%          7.000000
25%        10.000000
50%        12.000000
75%        14.000000
95%        18.000000
max        27

## 15. Preview final training examples

In [21]:
preview_cols = [
    "id",
    "category",
    "source",
    "headline_clean",
    "body_word_count_clean",
    "input_text",
    "target_text"
]

display(train_df[preview_cols].sample(5, random_state=RANDOM_STATE))

,id,category,source,headline_clean,body_word_count_clean,input_text,target_text
1044,train_01045,siyaasad,Goobjoog,Madaxweynaha Somaliland oo Joojiyey Munaasabad...,212,analyze somali news article: Madaxweynaha Soma...,category: siyaasad | headline: Madaxweynaha So...
439,train_00440,caafimaad,Goobjoog,Dowladda Soomaaliya oo Ku Dhawaaqday Toddobaad...,260,analyze somali news article: Guddoomiyaha gudd...,category: caafimaad | headline: Dowladda Sooma...
1729,train_01730,ciyaaro,Goobjoog,"Guddoomiye Warsame ""Wasiir Xamse Saciid Waxaan...",91,analyze somali news article: Guddoomiyihii Hor...,category: ciyaaro | headline: Guddoomiye Warsa...
296,train_00297,ciyaaro,Goobjoog,Kooxda Muqdisho City Club Oo Safar U Ah Puntland,236,analyze somali news article: Naadiga Dolwadda ...,category: ciyaaro | headline: Kooxda Muqdisho ...
2210,train_02211,ciyaaro,Laacibnet,Joe Cole: Xiddigo Waaweyn Oo Muhiim Ah Ayaa Ch...,210,analyze somali news article: Dhibaatada haysat...,category: ciyaaro | headline: Joe Cole: Xiddig...


## Final output files

The notebook creates this folder:

```text
final_combined_dataset/
```

Important files for mT5 fine-tuning:

```text
train.csv
validation.csv
test.csv
```

Use:

```text
input_text  -> model input
target_text -> model output
```